# Sesiones 10 y 11 — Evaluar sin engañarse y afinar sin trampa

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 4 — Evaluación y optimización

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar por qué **un solo** `train_test_split` puede engañar y usar **validación cruzada** para medir con promedio y dispersión.
2. **Comparar modelos** con validación cruzada y decidir si una diferencia es real o es un empate.
3. Reconocer la **fuga** de preprocesar (o balancear) antes de validar, y evitarla con un **`Pipeline`**.
4. Entrenar un **gradient boosting** y **buscar hiperparámetros** (grilla y azar) sin hacer trampa, sabiendo que el ganador de muchos intentos promete de más.
5. Guardar el modelo final con `joblib` y armar la **entrega final** de su proyecto.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                         ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: primera hora, medir bien (sin ver el examen); segunda hora, afinar sin engañarse.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje **bajo**) · 💥 **Trampa** · 📘 **Para entender** (lectura en casa) · ⚡ **Mini-reto**.

> **Datos:** Bank Marketing otra vez (Sesiones 5 y 6): 45 211 llamadas, 11,7 % "sí", sin `duration`. Así podemos comparar con lo que ya sabemos.
>
> **Hoy es una clase doble** (las Sesiones 10 y 11 completas siguen en el repositorio). Optuna y SHAP quedan como lectura en casa: [notebook completo de la Sesión 11](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S11/S11_estudiante.ipynb).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["modelo", "evaluación"], modulo=4, subtitulo="HOY: medir bien y afinar sin engañarse");

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

bank = pd.read_csv("bank.csv")
y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
print(X.shape, f"· 'sí' = {y.mean():.1%}")

> `get_dummies` va **antes** de partir y no es fuga: solo convierte cada categoría en columnas de 0 y 1, fila por fila, sin mirar la *y* ni calcular nada con otras filas. Lo que sí se filtra es lo que **aprende** de los datos (promedios, escalas, tasas de "sí"); eso vuelve en la sección 4.

## 1. La lotería del split

Hasta ahora siempre hicimos **un** `train_test_split` con `random_state=42`. ¿Qué pasa si cambiamos solo esa semilla? Un árbol de profundidad 5, 30 particiones distintas:

### 🔍 Predice
Con los 45 211 clientes, ¿cuánto cree que varía el AUC de prueba entre las 30 particiones? ¿Y si tuviéramos solo **1 500** clientes (el tamaño típico de un proyecto del curso)?

**Su predicción:** con 45 211: de ____ a ____ · con 1 500: de ____ a ____

In [ ]:
def auc_30_splits(X, y):
    scores = []
    for semilla in range(30):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=semilla, stratify=y)
        arbol = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr)
        scores.append(roc_auc_score(y_te, arbol.predict_proba(X_te)[:, 1]))
    return scores

pocos = bank.sample(1500, random_state=0).index
fg.loteria_split({"45 211 clientes": auc_30_splits(X, y), "1 500 clientes": auc_30_splits(X.loc[pocos], y.loc[pocos])});

## 2. Validación cruzada: muchos exámenes en vez de uno

Primero apartamos una **prueba** (20 %) que no vamos a tocar hasta el final. Con el 80 % restante hacemos **validación cruzada**: lo partimos en *k* pedazos (*folds*), entrenamos *k* veces, cada vez dejando un fold distinto para validar.

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Laboratorio 9 · Validación cruzada**](https://nitobest.github.io/curso-aad-python/laboratorio/09_validacion.html) · cinco exámenes en vez de uno, fold por fold; una partición contra k; el reto de duplicar los "sí" antes de validar. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
fg.cv_folds(k=5);

### 📐 Fundamento
Con $k$ folds se obtienen $k$ scores $s_1, \dots, s_k$. Se reporta su promedio y su dispersión:

$$\bar{s} = \frac{1}{k}\sum_{j=1}^{k} s_j \qquad \text{y} \qquad \text{desv}(s_1, \dots, s_k)$$

- El **promedio** es una estimación más estable que un solo split: usa todas las filas de entrenamiento para validar (cada una, una vez).
- La **desviación** dice qué tanto depende el resultado de la partición. Sin ella, un número no dice nada sobre su incertidumbre.
- **Estratificada** (`StratifiedKFold`): cada fold conserva la proporción de "sí" (11,7 %), igual que `stratify=y` en la Sesión 3.

<details><summary><b>Para quien quiera más:</b> ¿por qué no usar la desviación como un intervalo de confianza?</summary>

Los $k$ scores no son independientes (los modelos comparten la mayor parte del entrenamiento), así que la desviación **subestima** la incertidumbre real. Sirve para comparar modelos de forma informal; para una prueba formal se usan validación cruzada repetida (`RepeatedStratifiedKFold`) o pruebas pareadas corregidas (Nadeau y Bengio, 2003).
</details>

<details><summary><b>Detalle:</b> ¿por qué la misma desviación a veces sale un poco distinta?</summary>

En todo el curso la desviación de los folds es la de `scores.std()` de NumPy, que divide entre $k$ (`ddof=0`). La de pandas (`df.std()`) divide entre $k-1$ (`ddof=1`) y con 5 folds da un poco más. Ninguna es "la correcta"; lo importante es usar siempre la misma. Aquí usamos la de NumPy, también en las tablas.
</details>

![Dentro de cross_val_score](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S10/img/dentro_del_cross_val_score.png)

### 📘 Para entender — los tipos de validación cruzada y cuándo usar cada uno

**Qué es.** Partir el entrenamiento en $k$ pedazos (*folds*), entrenar $k$ veces dejando cada vez un pedazo distinto para validar, y reportar el promedio y la dispersión de los $k$ resultados. Cada fila se usa para validar exactamente una vez.

| Esquema | Cómo parte | Úselo cuando… | En scikit-learn |
|---|---|---|---|
| **K-fold** | $k$ pedazos al azar | regresión, filas independientes | `KFold(n_splits=5, shuffle=True)` |
| **K-fold estratificado** | cada pedazo con la misma proporción de clases | **clasificación** (siempre, y más con desbalance) | `StratifiedKFold(n_splits=5, shuffle=True)` |
| **Repetido** | repite el K-fold varias veces con otras semillas | pocos datos y quiere una estimación más estable | `RepeatedStratifiedKFold(n_splits=5, n_repeats=10)` |
| **Por grupos** | todas las filas de un mismo cliente, paciente o tienda en el mismo pedazo | hay varias filas por unidad | `GroupKFold`, `StratifiedGroupKFold` |
| **Por tiempo** | entrena con el pasado y valida con el bloque siguiente, avanzando | series de tiempo, pronósticos | `TimeSeriesSplit` |
| **Dejar uno fuera** | $k = n$: cada fila valida sola | muy pocos datos (decenas de filas) | `LeaveOneOut` (lento) |

**¿Cuántos folds?** 5 o 10 es el estándar. Más folds = cada modelo entrena con más datos, pero tarda más y los resultados se parecen más entre sí.

**Las tres funciones que va a usar:**

| Función | Devuelve | Para qué |
|---|---|---|
| `cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")` | un score por fold | lo más simple: una métrica |
| `cross_validate(modelo, X, y, cv=cv, scoring=[...])` | varias métricas y los tiempos | comparar en varias métricas a la vez |
| `cross_val_predict(modelo, X, y, cv=cv, method="predict_proba")` | una predicción por fila, hecha por el modelo que **no** la vio | armar una matriz de confusión o una curva con toda la validación |

**Los `scoring` más usados:** clasificación `"accuracy"`, `"roc_auc"`, `"f1"`, `"recall"`, `"precision"`, `"average_precision"`; regresión `"neg_mean_absolute_error"`, `"neg_root_mean_squared_error"`, `"r2"`. Los errores llevan `neg_` porque scikit-learn siempre **maximiza**: un MAE de 4 177 aparece como −4 177.

### ✏️ Completa — su primera validación cruzada
Calcule el AUC del árbol de profundidad 5 en cada uno de los 5 folds de `cv` (solo con el entrenamiento) y guárdelo en `scores_arbol`.

<details><summary>Pista</summary>

```python
scores_arbol = cross_val_score(arbol, X_train, y_train, cv=cv, scoring="roc_auc")
```
</details>

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
arbol = DecisionTreeClassifier(max_depth=5, random_state=42)
scores_arbol = ...   # TODO: AUC del árbol en cada fold de cv (solo entrenamiento)
print("AUC por fold:", scores_arbol.round(3))
print(f"promedio {scores_arbol.mean():.3f} ± {scores_arbol.std():.3f}")

### La perilla de la validación cruzada: `k`

In [ ]:
fg.cv_k_efecto(arbol, X_train, y_train, ks=(2, 3, 5, 10, 20));

El promedio casi no se mueve después de `k = 5`; el tiempo sí crece. Por eso el estándar es **5 o 10**. Con datasets pequeños conviene 10 (cada ronda entrena con más filas).

## 3. Comparar modelos de verdad

Ahora los cuatro modelos del Módulo 2 con la **misma** validación cruzada. La logística y k-NN necesitan escalar; para eso usamos `make_pipeline`, que pega el escalador al modelo (la sección 5 explica por qué esto importa tanto).

### 🔍 Predice
¿Qué orden espera (de mejor a peor AUC)? ¿Alguna diferencia será "empate"?

**Su predicción:** ____

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

modelos = {
    "árbol": arbol,
    "logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
}
resultados = {nombre: cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc") for nombre, m in modelos.items()}

### ✏️ Completa — agregue el bosque
Agregue a `resultados` la validación cruzada de un `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)`. Tarda unos 15 segundos (5 bosques de 200 árboles).

<details><summary>Pista</summary>

```python
bosque = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)
resultados["bosque"] = cross_val_score(bosque, X_train, y_train, cv=cv, scoring="roc_auc")
```
</details>

In [ ]:
bosque = ...                  # TODO: el RandomForestClassifier del enunciado
resultados["bosque"] = ...    # TODO: AUC por fold del bosque, con la misma cv
fg.cv_scores(resultados, titulo="Cuatro modelos, la misma validación cruzada (5 folds)");

### 📘 Para entender — ¿la diferencia entre dos modelos es real o es un empate?

Con validación cruzada, cada modelo tiene $k$ scores, no uno. Tres reglas prácticas para compararlos:

1. **Use la misma partición** (el mismo objeto `cv`) para todos los modelos. Así cada fold es una comparación justa: los dos modelos vieron exactamente los mismos datos.
2. **Mire las diferencias fold por fold.** Si un modelo le gana al otro en los 5 folds, la diferencia es consistente aunque sea pequeña. Si gana en 3 y pierde en 2, es ruido.
   ```python
   dif = resultados["bosque"] - resultados["logística"]
   print(dif.round(3), f"promedio {dif.mean():.3f} ± {dif.std():.3f}")
   ```
3. **Compare la diferencia con la dispersión.** Si la diferencia de promedios cabe varias veces en la desviación de los folds, es un empate.

**Ante un empate, elija por otra razón:** el modelo más **simple** de explicar, el más **rápido** de entrenar y de usar, o el más **estable** (menor desviación). Un 0,002 de AUC casi nunca justifica pasar de una logística que el banco entiende a un modelo que nadie puede explicar.

## 4. 💥 El atajo de duplicar los "sí"

En la Sesión 5 vimos que con 11,7 % de "sí" los modelos casi nunca dicen "sí". Un truco muy popular en internet: **sobremuestrear**, es decir, duplicar filas de "sí" al azar hasta que haya tantas como "no". Hagámoslo y validemos con nuestra flamante validación cruzada.

In [ ]:
from sklearn.utils import resample

entrenamiento = X_train.assign(y=y_train.values)
si, no = entrenamiento[entrenamiento["y"] == 1], entrenamiento[entrenamiento["y"] == 0]
balanceado = pd.concat([no, resample(si, replace=True, n_samples=len(no), random_state=0)])
X_bal, y_bal = balanceado.drop(columns="y"), balanceado["y"]
print(f"antes: {len(entrenamiento):,} filas, {y_train.mean():.1%} 'sí'  →  después: {len(balanceado):,} filas, {y_bal.mean():.0%} 'sí'")

### 🔍 Predice
El bosque sin trucos tenía AUC ≈ 0,79 y recall ≈ 0,19 (Sesión 6). Con los datos balanceados así, ¿qué AUC y qué recall promete la validación cruzada?

**Su predicción:** AUC ____ · recall ____

In [ ]:
from sklearn.model_selection import cross_validate

cv_dup = cross_validate(bosque, X_bal, y_bal, cv=cv, scoring=["roc_auc", "recall"])
print(f"Validación cruzada — AUC {cv_dup['test_roc_auc'].mean():.3f} · recall {cv_dup['test_recall'].mean():.3f}")

**¡AUC 0,96 y recall 0,89!** Un salto enorme. Antes de celebrar, hagamos lo que nunca hemos hecho en esta sesión: abrir la **prueba**, que el bosque no ha visto.

In [ ]:
from sklearn.metrics import recall_score

bosque_dup = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(X_bal, y_bal)
p = bosque_dup.predict_proba(X_test)[:, 1]
print(f"Prueba — AUC {roc_auc_score(y_test, p):.3f} · recall {recall_score(y_test, p >= 0.5):.3f}")

In [ ]:
fg.duplicados_fuga();

### La forma honesta
Si se quiere balancear, que sea **dentro** de cada ronda. La forma más simple ya la conocemos (Sesión 5): `class_weight="balanced"`, que pesa más los errores en "sí" sin copiar filas.

### ✏️ Completa — la versión honesta
Valide `bosque_bal` (el bosque con `class_weight="balanced"`) sobre el entrenamiento **sin duplicar**, con AUC y recall, y guarde el resultado en `cv_bal`. Después entrénelo con todo el entrenamiento y guarde en `p_bal` sus probabilidades de "sí" para la prueba.

<details><summary>Pista</summary>

```python
cv_bal = cross_validate(bosque_bal, X_train, y_train, cv=cv, scoring=["roc_auc", "recall"])
p_bal = bosque_bal.fit(X_train, y_train).predict_proba(X_test)[:, 1]
```
</details>

In [ ]:
bosque_bal = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)
cv_bal = ...   # TODO: validación cruzada de bosque_bal con AUC y recall
p_bal = ...    # TODO: probabilidades de "sí" en la prueba, entrenando con todo el entrenamiento
print(f"class_weight — validación cruzada: AUC {cv_bal['test_roc_auc'].mean():.3f} · recall {cv_bal['test_recall'].mean():.3f}")
print(f"class_weight — prueba:             AUC {roc_auc_score(y_test, p_bal):.3f} · recall {recall_score(y_test, p_bal >= 0.5):.3f}")
fg.promesa_vs_realidad({
    "duplicar los 'sí'\nantes de validar": {"CV": {"AUC": cv_dup["test_roc_auc"].mean(), "recall": cv_dup["test_recall"].mean()},
                                            "prueba": {"AUC": roc_auc_score(y_test, p), "recall": recall_score(y_test, p >= 0.5)}},
    "class_weight='balanced'\n(dentro de cada ronda)": {"CV": {"AUC": cv_bal["test_roc_auc"].mean(), "recall": cv_bal["test_recall"].mean()},
                                                         "prueba": {"AUC": roc_auc_score(y_test, p_bal), "recall": recall_score(y_test, p_bal >= 0.5)}},
});

### 📘 Para entender — las formas honestas de tratar el desbalance

| Técnica | Qué hace | Ventaja | Cuidado |
|---|---|---|---|
| **Mover el umbral** (S5) | el mismo modelo, otro corte de probabilidad | lo más simple; no toca el entrenamiento | elegir el umbral con validación, no con la prueba |
| **`class_weight="balanced"`** (S5, S6) | pesa más los errores en la clase rara | no copia filas: no hay fuga posible | no todos los modelos lo tienen (k-NN no) |
| **Submuestreo** | quita filas de la clase común | entrena más rápido | bota información |
| **Sobremuestreo / SMOTE** | duplica filas de la clase rara o fabrica filas "intermedias" (SMOTE) | a veces ayuda a modelos que no tienen `class_weight` | **solo dentro del Pipeline**, para que se haga en cada ronda con el entrenamiento de esa ronda |

Si de verdad necesita sobremuestrear, el paquete `imbalanced-learn` trae un `Pipeline` que solo lo aplica al entrenamiento de cada ronda:

```python
# pip install imbalanced-learn
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from imblearn.over_sampling import RandomOverSampler     # o SMOTE
modelo = make_pipeline_imb(RandomOverSampler(random_state=0), RandomForestClassifier(n_estimators=200, n_jobs=-1))
cross_validate(modelo, X_train, y_train, cv=cv, scoring=["roc_auc", "recall"])   # honesto
```

Y recuerde la lección de hoy: balancear casi no cambia el **AUC** (cómo ordena el modelo); lo que cambia es dónde queda el umbral efectivo y, con él, el recall.

## 5. `Pipeline`: el preprocesamiento viaja con el modelo

Un `Pipeline` encadena pasos (escalar → modelo) y se comporta como **un solo modelo**: `.fit()` ajusta todos los pasos con los datos que recibe, en orden; `.predict()` transforma y predice. Dentro de `cross_val_score`, eso significa que el escalador se re-ajusta **en cada ronda** solo con el entrenamiento de esa ronda.

### ✏️ Completa — su pipeline
Construya `mi_pipeline` (escalador estándar seguido de k-NN con 25 vecinos, como un solo objeto) y guarde en `res` su validación cruzada con tres métricas: AUC, recall y F1.

<details><summary>Pista</summary>

```python
mi_pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25))
res = cross_validate(mi_pipeline, X_train, y_train, cv=cv, scoring=["roc_auc", "recall", "f1"])
```
</details>

In [ ]:
mi_pipeline = ...   # TODO: escalador + k-NN (25 vecinos) en un solo objeto
res = ...           # TODO: su validación cruzada con AUC, recall y F1
por_fold = pd.DataFrame({m: res[f"test_{m}"] for m in ["roc_auc", "recall", "f1"]})
pd.DataFrame({"promedio": por_fold.mean(), "desv.": por_fold.std(ddof=0)}).T.round(3)   # ddof=0: la desviación de NumPy

`cross_validate` (con «e») devuelve varias métricas a la vez y los tiempos de entrenamiento. En la sección 7 la misma validación cruzada sirve para **elegir hiperparámetros** (grilla y azar) sin hacer trampa, y el modelo final de E4 se guarda como un `Pipeline` igual a este.

### 📘 Para entender — el `Pipeline`, a fondo

**Qué es.** Una cadena de pasos que scikit-learn trata como **un solo modelo**. Todos los pasos menos el último son **transformadores** (aprenden algo con `.fit()` y lo aplican con `.transform()`); el último es el modelo.

```
           Pipeline
 ┌──────────────┐   ┌─────────────────┐   ┌──────────────┐
 │ imputar      │ → │ escalar         │ → │ modelo       │
 │ (aprende la  │   │ (aprende media  │   │ (aprende los │
 │  mediana)    │   │  y desviación)  │   │  parámetros) │
 └──────────────┘   └─────────────────┘   └──────────────┘
 .fit(X_train, y_train): ajusta cada paso, en orden, solo con X_train
 .predict(X_nuevo): aplica cada paso con lo aprendido y predice
```

**Por qué usarlo siempre:**

1. **Evita la fuga**: dentro de la validación cruzada, cada paso se reajusta solo con el entrenamiento de cada ronda.
2. **Un solo objeto para producción**: se guarda y se carga entero (sección 10); el preprocesamiento viaja con el modelo.
3. **Permite buscar hiperparámetros de cualquier paso** (sección 7). Los nombres se arman con el paso y el parámetro separados por dos guiones bajos: `kneighborsclassifier__n_neighbors`.

**Con columnas de distintos tipos** se usa `ColumnTransformer`, que aplica un tratamiento a las numéricas y otro a las categóricas (lo verá completo en la sección 10):

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age", "balance", "campaign"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["job", "marital", "education"]),
])
modelo = make_pipeline(prep, LogisticRegression(max_iter=2000))
```

---
# Segunda hora — afinar sin engañarse

### Punto de control
Si algo falló en la primera hora, esta celda deja listos `X`, `y`, la partición y la validación cruzada `cv` de siempre. Ejecútela aunque todo le haya funcionado.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

bank = pd.read_csv("bank.csv")
y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_train.shape, X_test.shape)

### 🔍 Predice
En la sección 3 el bosque llegó a AUC **0,792** en validación cruzada. Con el modelo más usado en la industria y una búsqueda automática de hiperparámetros, ¿a cuánto cree que llegamos hoy?

**Su predicción:** AUC ≈ ____

## 6. Gradient boosting: árboles en fila

El bosque (Sesión 6) entrena cientos de árboles **en paralelo**, cada uno por su lado, y los promedia. El **gradient boosting** los entrena **en fila**: cada árbol nuevo, pequeño, se dedica a corregir lo que los anteriores todavía no explican (los **residuales**). Se ve mejor con una sola variable:

In [ ]:
fg.boosting_paso_a_paso(etapas=(1, 3, 10, 100));

### 📐 Fundamento
Después de $m$ árboles, la predicción es la suma de todos, cada uno con un peso pequeño $\eta$:

$$F_m(x) = F_{m-1}(x) + \eta \, h_m(x)$$

- $h_m$ es un árbol pequeño entrenado para predecir los residuales de $F_{m-1}$ (en general, el **gradiente** de la pérdida: por eso se llama *gradient* boosting).
- $\eta$ es el `learning_rate`, **el mismo de la Sesión 4**: pasos pequeños → hacen falta más árboles, pero se sobreajusta menos.
- En clasificación se suman *log-odds* y al final se pasa por la sigmoide, igual que la logística (Sesión 5).

<details><summary><b>Para quien quiera más:</b> HistGradientBoosting, LightGBM, XGBoost, CatBoost</summary>

Son implementaciones del mismo algoritmo con trucos de velocidad. `HistGradientBoostingClassifier` (scikit-learn) y LightGBM agrupan cada variable numérica en ~255 intervalos (histogramas), lo que los hace muy rápidos con muchas filas; XGBoost fue el que popularizó el método en Kaggle; CatBoost maneja muy bien las variables categóricas. Para este curso, `HistGradientBoostingClassifier` basta y no hay que instalar nada.
</details>

![Dentro del .fit() — gradient boosting](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S11/img/dentro_del_fit_boosting.png)

### 📘 Para entender — el gradient boosting, para qué sirve y cómo se configura

**Qué es.** Un conjunto de árboles pequeños entrenados **uno detrás de otro**: cada árbol nuevo se ajusta a lo que los anteriores todavía se equivocan, y su aporte se suma con un peso pequeño ($\eta$, el `learning_rate`). Es descenso de gradiente (Sesión 4), pero en vez de mover coeficientes, **agrega árboles**.

**Bosque vs boosting:**

| | Bosque aleatorio (S6) | Gradient boosting (hoy) |
|---|---|---|
| Cómo se entrenan los árboles | en **paralelo**, cada uno por su lado | en **fila**, cada uno corrige a los anteriores |
| Cómo son los árboles | profundos (sobreajustan solos) | pequeños (débiles solos) |
| Qué reduce | la **varianza** (promediando) | el **sesgo** (corrigiendo) |
| Riesgo | casi no sobreajusta al agregar árboles | **sí** sobreajusta con demasiados árboles: por eso la parada temprana |
| Ajuste | funciona bien de fábrica | gana un poco más ajustando, pero es más sensible a sus perillas |

**Las perillas principales** (`HistGradientBoostingClassifier` / `HistGradientBoostingRegressor`):

| Hiperparámetro | Qué controla | Para empezar |
|---|---|---|
| `learning_rate` | el tamaño de cada corrección | 0,05–0,1; más chico = más árboles, más estable |
| `max_iter` | número máximo de árboles | 500, con `early_stopping=True` |
| `max_leaf_nodes` / `max_depth` | tamaño de cada árbol | 15–31 hojas |
| `min_samples_leaf` | mínimo de filas por hoja | 20 (súbalo con datos ruidosos) |
| `l2_regularization` | castigo a las hojas extremas, como Ridge | 0–1 |
| `early_stopping=True` | detenerse cuando la validación interna deja de mejorar | siempre, en proyectos pequeños |

**Dónde se usa:** es el modelo que más gana en competencias de datos **en tabla** y uno de los más usados en la industria: *scoring* de crédito, detección de fraude, pronóstico de demanda, precios dinámicos, probabilidad de clic en publicidad, ordenar resultados de búsqueda. **No** es la mejor opción para imágenes, audio o texto libre (ahí mandan las redes neuronales).

| Implementación | Lo que la distingue |
|---|---|
| `HistGradientBoosting*` (scikit-learn) | sin instalar nada; maneja vacíos solo; muy rápido |
| LightGBM | muy rápido con millones de filas |
| XGBoost | el más popular históricamente; muchas opciones |
| CatBoost | maneja muy bien variables categóricas sin one-hot |

### ✏️ Completa — boosting con valores por defecto
Guarde en `scores_hgb` el AUC por fold de un `HistGradientBoostingClassifier(random_state=42)` con valores por defecto, con la misma validación cruzada de siempre (`cv`) y solo con el entrenamiento. Fíjese en el tiempo: el bosque tardaba ~15 s.

<details><summary>Pista</summary>

```python
scores_hgb = cross_val_score(HistGradientBoostingClassifier(random_state=42), X_train, y_train, cv=cv, scoring="roc_auc")
```
</details>

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier

t = time.time()
scores_hgb = ...   # TODO: AUC por fold del boosting por defecto (misma cv, solo entrenamiento)
print(f"boosting por defecto: {scores_hgb.mean():.3f} ± {scores_hgb.std():.3f}   ({time.time() - t:.1f} s)")

**0,796 ± 0,003** en unos segundos. ¿Le ganó al bosque de la sección 3 (0,792 ± 0,003)? Con la regla de la sección 3 no se puede afirmar: la diferencia (0,004) es del tamaño de la dispersión de los folds. En la sección 9 lo miramos con una lupa más fina (fold a fold). Donde el boosting gana sin discusión es en **tiempo**: segundos contra ~15 s del bosque.

¿Cuántos árboles usó? Lo decide solo, con **parada temprana**: aparta un 10 % del entrenamiento y deja de agregar árboles cuando esa validación interna deja de mejorar.

> ⚠️ **Para su proyecto:** con el valor por defecto (`early_stopping="auto"`) la parada temprana **solo se activa si hay más de 10 000 filas**. Con menos (lo normal en los proyectos del curso), escríbala explícitamente: `HistGradientBoostingClassifier(early_stopping=True, max_iter=500, random_state=42)`. Si no, entrena siempre 100 árboles (`max_iter=100`), sin mirar si sobran o faltan.

In [ ]:
hgb = HistGradientBoostingClassifier(early_stopping=True, max_iter=500, random_state=42).fit(X_train, y_train)
print("árboles que usó:", hgb.n_iter_)
fg.boosting_curva(hgb);

> 🧪 **¿Y XGBoost?** Es otra implementación del mismo gradient boosting, con frenos extra (regularización de las hojas, submuestreo de filas y columnas) y `scale_pos_weight` para el desbalance. En Bank Marketing da AUC ≈ 0,806, prácticamente lo mismo que el `HistGradientBoostingClassifier` de hoy. Para verlo funcionar paso a paso: [**Laboratorio 11 · Bootstrap, bosques y boosting**](https://nitobest.github.io/curso-aad-python/laboratorio/11_ensambles.html). En código cambia muy poco (Colab ya lo trae instalado):
>
> ```python
> from xgboost import XGBClassifier
> xgb = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, eval_metric="auc")
> cross_val_score(xgb, X_train, y_train, cv=cv, scoring="roc_auc")
> ```

## 7. Hiperparámetros: las perillas que el `.fit()` no aprende

Los **parámetros** los aprende el `.fit()` (los cortes de cada árbol, los coeficientes de la logística). Los **hiperparámetros** los decide usted antes: `learning_rate`, `max_leaf_nodes`, `k` en k-NN, `C` en la logística. Elegirlos es otra optimización: maximizar el AUC de **validación cruzada**, sin gradiente, donde cada evaluación cuesta 5 entrenamientos.

![Cómo buscar hiperparámetros](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S11/img/como_buscar_hiperparametros.png)

**Dentro del `.fit()` de una búsqueda** (la misma ficha de siempre, un nivel más arriba):

| | Búsqueda de hiperparámetros |
|---|---|
| Qué minimiza | el error de **validación cruzada** (aquí: maximiza el AUC promedio de los folds) |
| Moviendo qué | los **hiperparámetros** θ (`learning_rate`, `max_leaf_nodes`…); los parámetros los sigue aprendiendo cada `.fit()` interno |
| Cómo lo busca | sin gradiente: grilla (todas las combinaciones), azar (muestras) u Optuna (aprende dónde le ha ido bien; en casa) |
| Perilla clave | el **presupuesto** (`n_iter`, `n_trials`) y los **rangos**; también cuántos folds |

### La grilla
`GridSearchCV` prueba todas las combinaciones de una lista. Aquí 3 × 3 = 9 combinaciones × 5 folds = 45 entrenamientos. El código va completo: lo importante es leer el resultado.

In [ ]:
from sklearn.model_selection import GridSearchCV

grilla = {"learning_rate": [0.03, 0.1, 0.3], "max_leaf_nodes": [7, 15, 31]}
busqueda = GridSearchCV(HistGradientBoostingClassifier(random_state=42), grilla, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print("mejor combinación:", busqueda.best_params_, f"· AUC {busqueda.best_score_:.4f}")

### 🔍 Predice
La mejor casilla de la grilla dio el AUC de arriba. De las **9** casillas, ¿cuántas cree que quedaron a menos de 0,005 de la mejor?

**Su predicción:** ____ de 9

In [ ]:
fg.grid_heatmap(busqueda.cv_results_, "learning_rate", "max_leaf_nodes");

**Lectura:** la mejor casilla (0,797) le gana a la configuración por defecto (0,796) por **0,001**, y seis de las nueve casillas están a menos de 0,005. Los valores por defecto de scikit-learn ya son buenos; la grilla casi no encontró nada.

Fíjese en la esquina de `learning_rate = 0.03` con 7 hojas: es la peor casilla (0,780). No es que los pasos pequeños sean malos (con 31 hojas, el mismo 0,03 llega a 0,797): es que la grilla deja `max_iter` en 100 árboles, y con pasos de 0,03 y árboles pequeños cien árboles **no alcanzan** a llegar. Una perilla castiga a la otra; con `early_stopping=True` y un `max_iter` grande, el modelo decide solo cuántos árboles necesita cada `learning_rate`.

### ¿Y la búsqueda al azar?
Con el mismo presupuesto, `RandomizedSearchCV` prueba combinaciones al azar. Parece peor, pero suele ganar (Bergstra y Bengio, 2012):

In [ ]:
fg.grid_vs_random();

### 📘 Para entender — grilla, azar u Optuna: cómo elegir

| Método | Cómo busca | Úselo cuando… | En código |
|---|---|---|---|
| **Grilla** | prueba **todas** las combinaciones de una lista | 1–2 perillas con pocos valores; quiere un mapa completo | `GridSearchCV(modelo, {"param": [..]}, cv=cv)` |
| **Al azar** | prueba combinaciones sorteadas de unos rangos | 3 o más perillas; presupuesto fijo | `RandomizedSearchCV(modelo, distribuciones, n_iter=40, cv=cv)` |
| **Bayesiana (Optuna, TPE)** | usa los intentos anteriores para proponer el siguiente | cada intento es caro y hay varias perillas | `optuna.create_study(direction="maximize")` |

**¿Por qué el azar le gana a la grilla con el mismo presupuesto?** Casi siempre solo una o dos perillas importan de verdad. Con una grilla de 3 × 3 solo se prueban **3** valores de la perilla importante; con 9 intentos al azar se prueban **9** valores distintos de cada una.

**Buenas prácticas:**

1. **Escala logarítmica** para perillas que se mueven en órdenes de magnitud (`learning_rate`, `alpha`, `C`, `l2_regularization`): probar 0,001; 0,01; 0,1 tiene más sentido que 0,1; 0,2; 0,3.
   ```python
   from scipy.stats import loguniform, randint
   distribuciones = {"learning_rate": loguniform(0.01, 0.3), "max_leaf_nodes": randint(7, 64)}
   ```
2. **Busque sobre el `Pipeline` completo** para no filtrar nada; los nombres llevan el paso: `{"histgradientboostingclassifier__learning_rate": ...}`.
3. **Presupuesto proporcional a los datos**: con pocas filas, pocas perillas y pocos intentos (lo verá en la sección 8).
4. **Fije la semilla** de la búsqueda y del modelo para poder repetir el resultado.

## 8. 💥 El mejor de muchos sorteos

`best_score_` es **el máximo** entre muchas configuraciones, cada una medida con ruido. El máximo de muchas mediciones ruidosas siempre sale optimista. Con 36 000 filas el ruido es pequeño; con un dataset del tamaño de su proyecto, no.

Hagamos lo que haría cualquiera en su proyecto: 2 000 clientes, se aparta el 30 % y con los otros **1 400** se prueban 60 configuraciones al azar y se queda con la mejor. Después, algo que en su proyecto no puede hacer: medir **todas** en **8 000 clientes del banco que nadie usó**. Con tantas filas, esa medición es casi la verdad de cada configuración.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

pocos = bank.sample(2000, random_state=1).index
Xp_train, _, yp_train, _ = train_test_split(X.loc[pocos], y.loc[pocos], test_size=0.3, random_state=42, stratify=y.loc[pocos])
nuevos = X.drop(index=pocos).sample(8000, random_state=0).index      # 8 000 clientes que no entraron a nada: la "verdad"
distribuciones = {"learning_rate": loguniform(0.01, 0.3), "max_leaf_nodes": randint(4, 64), "min_samples_leaf": randint(5, 200),
                  "l2_regularization": loguniform(1e-3, 10), "max_iter": randint(50, 400)}
azar = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=42), distribuciones, n_iter=60, cv=cv,
                          scoring="roc_auc", random_state=0, n_jobs=-1).fit(Xp_train, yp_train)
print(f"la ganadora de 60 configuraciones: AUC {azar.best_score_:.3f} en validación cruzada")

### 🔍 Predice
La ganadora prometió **0,770**. En los 8 000 clientes nuevos, ¿cuánto dará? ¿Seguirá siendo la mejor de las 60? Y una configuración cualquiera (no la ganadora), ¿también baja?

**Su predicción:** ganadora ≈ ____ · puesto ____ de 60 · ¿las demás también bajan? ____

In [ ]:
def auc_en_nuevos(modelo):
    """Entrena con los mismos 1 400 clientes y mide el AUC en los 8 000 que nadie usó."""
    return roc_auc_score(y.loc[nuevos], modelo.fit(Xp_train, yp_train).predict_proba(X.loc[nuevos])[:, 1])

resultados = pd.DataFrame(azar.cv_results_)
resultados["nuevos"] = [auc_en_nuevos(HistGradientBoostingClassifier(**p, random_state=42)) for p in resultados["params"]]
caida = resultados["mean_test_score"] - resultados["nuevos"]
por_defecto = HistGradientBoostingClassifier(random_state=42)
defecto = (cross_val_score(por_defecto, Xp_train, yp_train, cv=cv, scoring="roc_auc").mean(), auc_en_nuevos(por_defecto))
g = azar.best_index_
print(f"ganadora:    validación {resultados['mean_test_score'][g]:.3f} → nuevos {resultados['nuevos'][g]:.3f}  (cae {caida[g]:.3f}; "
      f"puesto {(resultados['nuevos'] > resultados['nuevos'][g]).sum() + 1} de {len(resultados)} en los datos nuevos)")
print(f"las 60:      caída típica (mediana) {caida.median():.3f}")
print(f"por defecto: validación {defecto[0]:.3f} → nuevos {defecto[1]:.3f}  (cae {defecto[0] - defecto[1]:.3f})")
fg.ganador_busqueda(resultados["mean_test_score"], resultados["nuevos"], g, defecto=defecto, n_clientes=1400, n_nuevos=8000);

### 💥 Lo que pasó

<details><summary>¿Por qué pasó esto? (ábralo después de la discusión)</summary>

Hay **dos** caídas distintas en la figura, y conviene no confundirlas:

1. **Toda la nube cae bajo la diagonal** (caída típica ≈ 0,032). Le pasa a casi todas, también a la configuración por defecto, que nadie eligió (0,740 → 0,715). Eso **no** es la maldición del ganador: es la lotería de la sección 1. Con solo 1 400 clientes, la propia validación cruzada es ruidosa, y esta muestra resultó un poco más "fácil" que el resto del banco.
2. **Lo propio de la ganadora**: prometió **0,770** y en datos nuevos dio **0,714**: cayó **0,057**, más que la típica, y quedó en el **puesto 32 de 60**. Su primer lugar en validación era en buena parte **suerte**. Eso sí es la **maldición del ganador**: si se elige el máximo de muchas mediciones ruidosas, ese máximo sobreestima. Cuantos más intentos y menos datos, peor.

**No** es una fuga como las de antes: nada de los datos nuevos se usó para elegir.

**Qué hacer en su proyecto:**
1. Reportar como desempeño final el de la **prueba** (una sola vez), no el `best_score_`.
2. Con pocos datos, preferir pocas perillas y rangos razonables a 500 intentos de Optuna.
3. No esperar que la prueba confirme el orden fino entre configuraciones: diferencias de milésimas en validación no significan nada.

<details><summary><b>Para quien quiera más:</b> validación cruzada anidada</summary>

Si necesita una estimación honesta del procedimiento completo sin gastar la prueba: **validación cruzada anidada**. La búsqueda entera (las 60 configuraciones con su propia validación cruzada) se repite **dentro** de cada fold de otra validación cruzada externa, y se reporta el promedio de los folds externos. Cuesta k veces más cómputo:

```python
from sklearn.model_selection import cross_val_score
cross_val_score(azar, Xp_train, yp_train, cv=5, scoring="roc_auc")   # cada fold externo hace su propia búsqueda
```
</details>

En la Sesión 4 lo vimos en pequeño: decenas de variantes que no le ganaban a la regresión más simple.
</details>

## 9. La comparación honesta (con todos los datos)

De la línea base al boosting afinado con la grilla de la sección 7: validación cruzada (5 folds, la misma `cv` para todos) y **una** evaluación en prueba de cada uno.

> **Sobre la prueba:** el modelo final se eligió con la **validación cruzada**, antes de abrir la prueba. Aquí mostramos la prueba de los cinco solo con fines didácticos; en su proyecto se abre una vez, para el modelo final.

### ✏️ Completa — el modelo afinado
Cree `afinado`: un boosting con los mejores hiperparámetros que encontró la grilla (`busqueda.best_params_`) y `random_state=42`.

<details><summary>Pista</summary>

```python
afinado = HistGradientBoostingClassifier(**busqueda.best_params_, random_state=42)
```
</details>

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

afinado = ...   # TODO: boosting con los mejores hiperparámetros de la grilla y random_state=42
candidatos = {
    "línea base": DummyClassifier(strategy="prior"),
    "logística (S5)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "bosque (S6)": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "boosting\npor defecto": HistGradientBoostingClassifier(random_state=42),
    "boosting\nafinado (grilla)": afinado,
}
filas, por_fold = {}, {}
for nombre, m in candidatos.items():
    por_fold[nombre] = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    filas[nombre] = (por_fold[nombre].mean(), por_fold[nombre].std(), roc_auc_score(y_test, m.fit(X_train, y_train).predict_proba(X_test)[:, 1]))
fg.comparacion_final(filas)
pd.DataFrame(filas, index=["CV promedio", "CV desv.", "prueba"]).T.round(3)

¿Las diferencias pequeñas son reales? Como todos usaron **la misma** `cv`, se pueden restar **fold a fold** (diferencias pareadas): si la diferencia cambia de signo entre folds, o es del tamaño de su dispersión, es empate.

In [ ]:
for a, b in [("boosting\npor defecto", "bosque (S6)"), ("boosting\nafinado (grilla)", "boosting\npor defecto")]:
    d = por_fold[a] - por_fold[b]
    print(f"{a.replace(chr(10), ' ')} − {b.replace(chr(10), ' ')}: por fold {d.round(3)} → {d.mean():+.4f} ± {d.std():.4f}")

## 10. Guardar el modelo y usarlo con datos nuevos

En un proyecto real el modelo se usa después, con clientes nuevos que llegan **tal como vienen** (texto, sin *one-hot*). Por eso el modelo final se guarda como un `Pipeline` que incluye el preprocesamiento.

**La regla del curso para el modelo que se entrega:** primero se **evalúa** (validación cruzada para elegir, una sola vez la prueba para reportar); después, con los hiperparámetros ya fijos, se **reentrena con todas las filas** (entrenamiento + prueba) y ese es el que se guarda. El número que se reporta sigue siendo el de la prueba. Anote también la versión de scikit-learn: un `.joblib` solo se carga bien con la misma versión.

In [ ]:
import joblib
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder

crudas = bank.drop(columns=["deposit", "duration"])
categoricas = crudas.select_dtypes(exclude="number").columns.tolist()
modelo_final = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas), remainder="passthrough"),
    HistGradientBoostingClassifier(random_state=42),     # el por defecto: afinar no ganó nada defendible (sección 9)
)
modelo_final.fit(crudas, y)          # ya evaluado (CV y una prueba): el que se ENTREGA se reentrena con todas las filas
joblib.dump(modelo_final, "modelo_bank.joblib")
import sklearn
from pathlib import Path
print("guardado:", Path("modelo_bank.joblib").stat().st_size // 1024, "KB · scikit-learn", sklearn.__version__)

In [ ]:
cargado = joblib.load("modelo_bank.joblib")
cliente_nuevo = pd.DataFrame([{"age": 35, "job": "management", "marital": "single", "education": "tertiary", "default": "no",
                               "balance": 2500, "housing": "no", "loan": "no", "contact": "cellular", "day": 15, "month": "oct",
                               "campaign": 1, "pdays": 90, "previous": 2, "poutcome": "success"}])
print(f"probabilidad de que abra el depósito: {cargado.predict_proba(cliente_nuevo)[0, 1]:.0%}")

Un archivo, una línea para cargarlo, y el modelo recibe al cliente con sus columnas originales. Eso es lo que se entrega a quien lo va a usar (una aplicación, un tablero, otro equipo).

## 11. 📦 Entrega final del proyecto (reemplaza E3 y E4)

Sobre **su** dataset (el de E1–E2), en un notebook de Colab. **Fecha:** el profesor la confirma en clase. El lunes 30 cada persona presenta en **2 minutos con 1 lámina**: la pregunta, el mejor modelo frente a la línea base, la métrica y una conclusión.

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Pregunta y métrica | La *y*, la métrica elegida y por qué (qué error duele más) |
| 2 | Prueba apartada | `train_test_split` al inicio; no se toca hasta el punto 5 |
| 3 | `Pipeline` | Todo el preprocesamiento dentro (codificar, escalar, imputar, balancear) |
| 4 | Comparación con validación cruzada | Línea base + **al menos 2 modelos**, uno de ellos bosque o boosting; promedio ± desviación |
| 5 | **Una** evaluación en prueba | Del modelo elegido; compárela con su validación cruzada. En clasificación, el umbral se elige en validación (no en prueba) |
| 6 | Lectura honesta | ¿Cuánto le gana a la línea base? ¿La diferencia entre modelos es real o empate? ¿Qué limita el desempeño? |
| 7 | Modelo guardado | El `Pipeline` final reentrenado con todas las filas (sección 10), guardado con `joblib` y una celda que lo carga y predice una fila nueva |

**Opcionales (suman):** búsqueda de hiperparámetros (sección 7) · SHAP o importancias · PCA o clustering sobre sus datos (lo que fue la entrega E3, se ve el 30).

Rúbrica (1–4 cada uno): corrección técnica · ausencia de fuga · honestidad en el reporte (prueba frente a validación) · claridad. Ejemplo completo resuelto: [guía end-to-end (hoteles)](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/proyecto/guia_end_to_end_hoteles.ipynb).

Plantilla mínima (cambie lo que está en MAYÚSCULAS). Sirve para **clasificación** y para **regresión**: cambie `TAREA`.

In [ ]:
# ---------- PLANTILLA DE LA ENTREGA FINAL ----------
# import pandas as pd, numpy as np, joblib, sklearn
# from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
# from sklearn.pipeline import make_pipeline
# from sklearn.compose import make_column_transformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.linear_model import LogisticRegression, Ridge
# from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
# from sklearn.dummy import DummyClassifier, DummyRegressor
#
# TAREA = "clasificacion"                     # o "regresion"
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# y = df["SU_Y"];  X = df.drop(columns=["SU_Y", "COLUMNAS_DEL_FUTURO", "IDENTIFICADORES"])
# if TAREA == "clasificacion":
#     METRICA = "roc_auc"                                     # o "recall", "f1"… la que justificó en el punto 1
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     modelos = {"línea base": DummyClassifier(strategy="prior"),
#                "logística": LogisticRegression(max_iter=2000, class_weight="balanced"),
#                "bosque": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=42),
#                "boosting": HistGradientBoostingClassifier(early_stopping=True, max_iter=500, random_state=42)}
# else:
#     METRICA = "neg_root_mean_squared_error"                 # sale con signo menos: el RMSE es ese número sin el signo
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   # sin stratify
#     cv = KFold(n_splits=5, shuffle=True, random_state=42)
#     modelos = {"línea base": DummyRegressor(), "ridge": Ridge(),
#                "bosque": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42),
#                "boosting": HistGradientBoostingRegressor(early_stopping=True, max_iter=500, random_state=42)}
#
# categoricas = X.select_dtypes(exclude="number").columns.tolist()
# numericas = X.select_dtypes(include="number").columns.tolist()
# def prep():   # un preprocesador nuevo para cada modelo: se ajusta DENTRO de cada fold
#     return make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
#                                    (StandardScaler(), numericas))
#
# tabla = {}
# for nombre, m in modelos.items():
#     s = cross_val_score(make_pipeline(prep(), m), X_train, y_train, cv=cv, scoring=METRICA)
#     tabla[nombre] = (s.mean(), s.std())
# print(pd.DataFrame(tabla, index=["CV promedio", "CV desv."]).T.round(3))
#
# ELEGIDO = "boosting"                                        # el que gane en validación cruzada (o el más simple si empatan)
# final = make_pipeline(prep(), modelos[ELEGIDO]).fit(X_train, y_train)
# from sklearn.metrics import get_scorer
# print("prueba:", get_scorer(METRICA)(final, X_test, y_test))   # ← ESTE es el número que se reporta (una sola vez)
#
# # Modelo que se entrega: el mismo, reentrenado con TODAS las filas; guardado en su Drive
# final.fit(X, y)
# from google.colab import drive; drive.mount("/content/drive")
# joblib.dump(final, "/content/drive/MyDrive/SU_CARPETA/mi_modelo.joblib")
# print("scikit-learn", sklearn.__version__)   # anótelo: el .joblib se carga con esta versión

## 12. Cierre — qué cambió hoy en el pipeline

**Lo que se lleva del Módulo 4:**
- Un número sin su incertidumbre no dice nada: **validación cruzada**, promedio ± desviación (secciones 1–3).
- Todo lo que aprende de los datos (escalar, balancear, codificar) va **dentro del `Pipeline`**; si no, la validación promete de más (secciones 4–5).
- El **mejor de muchos intentos es optimista**: el número que se reporta es el de la prueba, abierta una sola vez (sección 8).
- Casi toda la ganancia viene de **buenos datos y un buen tipo de modelo**, no de afinar perillas (sección 9).

**Próxima y última clase (lunes 30):** primera hora, **Módulo 3 exprés** (K-means, la trampa de no escalar, silueta y PCA); segunda hora, **presentaciones relámpago** de 2 minutos con 1 lámina.

**Para profundizar en casa:** [Sesión 10 completa](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S10/S10_estudiante.ipynb) (incluye el mini-reto de "la otra fuga famosa") · [Sesión 11 completa](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S11/S11_estudiante.ipynb) (Optuna, SHAP y PyCaret) · [Laboratorio 9 · Validación cruzada](https://nitobest.github.io/curso-aad-python/laboratorio/09_validacion.html) · [Laboratorio 11 · Bosques y boosting](https://nitobest.github.io/curso-aad-python/laboratorio/11_ensambles.html).

### 📘 La Sesión 10 en una página (para repasar)

| Concepto | Qué es | En código |
|---|---|---|
| Lotería del split | un solo `train_test_split` puede dar un número muy distinto con otra semilla | con pocos datos, nunca confíe en un solo número |
| Validación cruzada | $k$ rondas de entrenar y validar | `cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc")` |
| Estratificada | cada fold con la misma proporción de clases | `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` |
| Promedio ± desviación | el número y su incertidumbre | `scores.mean()`, `scores.std()` |
| Empate | diferencia que cabe en la dispersión | elegir el más simple |
| Fuga en la validación | preprocesar antes de partir en folds | todo dentro del `Pipeline` |
| `Pipeline` | pasos de preprocesamiento + modelo como uno solo | `make_pipeline(StandardScaler(), modelo)` |
| Prueba | se abre **una** vez, al final | reportar CV ± y prueba juntos |

### 📘 La Sesión 11 en una página (para repasar)

| Concepto | Qué es | En código |
|---|---|---|
| Gradient boosting | árboles pequeños en fila que corrigen los errores anteriores | `HistGradientBoostingClassifier(early_stopping=True, max_iter=500)` |
| Parada temprana | detenerse cuando la validación interna deja de mejorar | `early_stopping=True` (explícito con menos de 10 000 filas) |
| Hiperparámetro | perilla que decide usted antes del `.fit()` | `learning_rate`, `max_leaf_nodes`… |
| Grilla / azar | probar todas las combinaciones / combinaciones sorteadas | `GridSearchCV`, `RandomizedSearchCV` |
| Optuna | búsqueda que aprende de los intentos anteriores | `study.optimize(objetivo, n_trials=...)` |
| Maldición del ganador | el mejor de muchos intentos ruidosos es optimista | reportar la prueba, no `best_score_` |
| Validación anidada | la búsqueda completa dentro de cada fold externo | `cross_val_score(busqueda, X, y, cv=5)` |
| SHAP | reparte cada predicción entre las variables | `shap.TreeExplainer(modelo)(X)` |
| Modelo entregado | `Pipeline` reentrenado con todas las filas | `joblib.dump(modelo, "modelo.joblib")` |